(course-core-19)=

# Module 19: Structures and Trajectories

**Welcome back, Apprentice Master.** In [Module 18: Extracting and Removing Elements](../Common_Core/18_Extracting_and_Removing_Elements.ipynb), you mastered isolating sub-systems using `msm.extract()` and `msm.remove()`. Now we focus on the temporal and spatial ensemble dimension of molecular systems: **Structures and Trajectories**.

Whether your system originates from a molecular dynamics trajectory, an NMR conformational ensemble, or a series of Monte Carlo snapshots, MolSysMT treats all 3D conformers under the universal technical term **`structure`** (with invariant coordinate tensor shape `(n_structures, n_atoms, 3)`). This module explores structural metadata, targeted disk slicing, and memory-efficient streaming for heavy trajectories.

(course-core-19-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Inspect structural dimensions (`n_structures`, `box`, `time`).
- Perform targeted disk slicing using `structure_indices` in `msm.get()` without loading whole trajectories into memory.
- Stream trajectory selections in chunks using `msm.Iterator()` for memory-efficient processing.
- Understand why MolSysMT uses the universal technical term **`structure`** instead of "frame".
```

### 1. Structural Invariants

Let's begin by loading our Villin Headpiece demonstration trajectory system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load Villin Headpiece trajectory system (topology + trajectory DCD)
villin_topo = systems['chicken villin HP35']['chicken_villin_HP35_solvated.h5msm']
villin_traj = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']
sys = [villin_topo, villin_traj]

# Query total number of structures in the trajectory
n_structs = msm.get(sys, element='system', n_structures=True)
print(f"Total structures in trajectory: {n_structs}")

Total structures in trajectory: 20


:::{note}
:class: dropdown
**Terminology Note: "Structure" vs. "Frame"**  
In MolSysMT, the universal technical term for a 3D spatial conformation is **`structure`** (`n_structures`, `structure_indices`, array shape `(n_structures, n_atoms, 3)`). While "frame" is natural in molecular dynamics contexts, "structure" encompasses NMR ensembles and non-sequential conformational sets.
:::

### 2. Targeted Slicing

Loading multi-gigabyte trajectory files into RAM can quickly exhaust system memory. Using `msm.get()` with **`structure_indices`** allows you to read specific coordinate snapshots directly from disk without converting or loading the full trajectory:

In [2]:
# Extract coordinates for the first structure directly from disk
coords_first = msm.get(sys, structure_indices=0, coordinates=True)
print(f"Coordinates shape for 1 structure: {coords_first.shape}")

# Extract coordinates for targeted structures (first, middle, and last)
coords_subset = msm.get(sys, structure_indices=[0, 10, 19], coordinates=True)
print(f"Coordinates shape for targeted subset (3 structures): {coords_subset.shape}")

Coordinates shape for 1 structure: (1, 4369, 3)


Coordinates shape for targeted subset (3 structures): (3, 4369, 3)


### 3. Streaming Trajectories

When analyzing large trajectory datasets, loading all coordinates into RAM at once is inefficient. Use **`msm.Iterator()`** with `structure_indices='all'` and the `chunk` parameter to stream selections in small batches for memory-safe processing:

In [3]:
# Initialize an iterator over protein Alpha-Carbons in chunks of 5 structures
iterator = msm.Iterator(sys, selection='atom_name == "CA"', chunk=5, structure_indices='all', coordinates=True)

# Iterate through trajectory selection chunks without memory overload
for batch_idx, coords in enumerate(iterator):
    print(f"Batch {batch_idx + 1}: extracted coordinates shape = {coords.shape}")

  _prune_structural_attributes_off_the_axis(molecular_system, from_forms, from_attributes)



Batch 1: extracted coordinates shape = (5, 36, 3)


Batch 2: extracted coordinates shape = (5, 36, 3)


Batch 3: extracted coordinates shape = (5, 36, 3)


Batch 4: extracted coordinates shape = (5, 36, 3)


--- 

### 🏆 Challenge 19: The Trajectory Master

1. Load the Villin Headpiece trajectory system (`[villin_topo, villin_traj]`).
2. Query the total number of structures using `msm.get(sys, element='system', n_structures=True)`.
3. Extract coordinates for targeted structure indices `[0, 5, 15]` using `structure_indices`.
4. Stream the protein Alpha-Carbons (`selection='atom_name == "CA"'`) in chunks of 5 structures using `msm.Iterator()`.

Mastering structures and trajectory streaming completes your core analytical foundation. In [Module 20: The Specialized Domains](../Common_Core/20_The_Specialized_Domains.ipynb), we will preview the specialized master paths of MolSysMT.

```{key-takeaway}
MolSysMT treats all 3D conformers under the universal technical term `structure` with shape `(n_structures, n_atoms, 3)`. Trajectories can be sliced on demand using `structure_indices` or streamed memory-safely in chunks using `msm.Iterator()`.
```

(course-core-19-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.get` — Attribute extraction engine.
- {func}`molsysmt.basic.Iterator` — Memory-efficient streaming iterator.

**Related Course Modules & Guides:**
- Previous Module: [Module 18: Extracting and Removing Elements](../Common_Core/18_Extracting_and_Removing_Elements.ipynb)
- Next Module: [Module 20: The Specialized Domains](../Common_Core/20_The_Specialized_Domains.ipynb)
- User Guide: {ref}`user-foundations`
:::